In [ ]:
import ccxt
import pandas as pd
import time
from datetime import datetime, timedelta

# ── ① 取得期間指定（YYYY-MM-DD）
start_date = '2025-05-01'
end_date   = '2025-06-'

symbol = 'ADA/USDT'
timeframe = '1m'

# ── ② ccxt で ISO → タイムスタンプ変換
exchange = ccxt.mexc()
start_iso = start_date + 'T00:00:00Z'
start_ts = exchange.parse8601(start_iso)

end_dt = datetime.fromisoformat(end_date) + timedelta(days=1)
end_iso = end_dt.isoformat() + 'Z'
end_ts = exchange.parse8601(end_iso)

def fetch_ohlcv_range(symbol, timeframe, since_ts, until_ts):
    all_data = []
    since = since_ts
    limit = 500
    while since < until_ts:
        candles = exchange.fetch_ohlcv(symbol, timeframe, since=since, limit=limit)
        if not candles:
            break
        all_data.extend(candles)
        last_ts = candles[-1][0]
        if last_ts >= until_ts:
            break
        since = last_ts + 1
        time.sleep(exchange.rateLimit / 1000)
    return [c for c in all_data if c[0] < until_ts]

if __name__ == '__main__':
    ohlcv = fetch_ohlcv_range(symbol, timeframe, start_ts, end_ts)
    df = pd.DataFrame(ohlcv,
                      columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
    # timestamp → datetime に変換
    df['datetime'] = pd.to_datetime(df['timestamp'], unit='ms')
    # 不要な timestamp 削除
    df = df.drop(columns=['timestamp'])
    # カラム並べ替え（datetime を先頭）
    df = df[['datetime', 'open', 'high', 'low', 'close', 'volume']]

    # ファイル名に期間・時間足を反映
    sym = symbol.replace('/', '').lower()
    filename = f"{sym}_{start_date}_to_{end_date}_{timeframe}.csv"

    # CSV 出力
    df.to_csv(filename, index=False)
    print(f"OHLCV データを {len(df)} 行取得しました：")
    print(f"  {df['datetime'].iloc[0]} 〜 {df['datetime'].iloc[-1]}")
    print(f"出力ファイル: {filename}")


OHLCV データを 366 行取得しました：
  2024-06-17 00:00:00 〜 2025-06-17 00:00:00
出力ファイル: adausdt_2024-06-17_to_2025-06-17_1d.csv


In [1]:
import ccxt
import pandas as pd
import time
from datetime import datetime
import os

exchange = ccxt.mexc()
symbol = 'ADA/USDC'
output_dir = 'ohlcv_data'
os.makedirs(output_dir, exist_ok=True)

# 時間足ごとの推奨日付区間（start_date, end_date）
timeframes_config = {
    '5m':  ('2025-05-11', '2025-06-26'),
    '15m': ('2025-05-01', '2025-06-24'),
    '1h':  ('2025-03-24', '2025-06-24'),
    '4h':  ('2024-12-21', '2025-06-21'),
    '1d':  ('2024-06-17', '2025-06-17'),
}

def iso_ts(date_str):
    return int(exchange.parse8601(date_str + 'T00:00:00Z'))

def fetch_ohlcv_range(symbol, timeframe, since_ts, until_ts):
    all_data = []
    limit = 1000
    since = since_ts
    while since < until_ts:
        try:
            candles = exchange.fetch_ohlcv(symbol, timeframe, since=since, limit=limit)
        except Exception as e:
            print(f"[{timeframe}] Error: {e}")
            break
        if not candles:
            break
        all_data.extend(candles)
        last_ts = candles[-1][0]
        if last_ts >= until_ts:
            break
        since = last_ts + 1
        time.sleep(exchange.rateLimit / 1000)
    return [c for c in all_data if c[0] < until_ts]

# データ取得ループ
for tf, (start_date, end_date) in timeframes_config.items():
    print(f"▶ {tf} を取得中: {start_date} ～ {end_date}")
    since_ts = iso_ts(start_date)
    until_ts = iso_ts(end_date)

    ohlcv = fetch_ohlcv_range(symbol, tf, since_ts, until_ts)
    if not ohlcv:
        print(f"⚠ データが取得できませんでした: {symbol} {tf}")
        continue

    df = pd.DataFrame(ohlcv, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
    df['datetime'] = pd.to_datetime(df['timestamp'], unit='ms')
    df = df[['datetime', 'open', 'high', 'low', 'close', 'volume']]

    filename = f"{output_dir}/{symbol.replace('/', '').lower()}_{start_date}_to_{end_date}_{tf}.csv"
    df.to_csv(filename, index=False)
    print(f"✅ {len(df)} 行取得済: {filename}")
    print(f"  {df['datetime'].iloc[0]} ～ {df['datetime'].iloc[-1]}\n")

print("🎉 全時間足の取得が完了しました。")


▶ 5m を取得中: 2025-05-11 ～ 2025-06-26
✅ 13248 行取得済: ohlcv_data/adausdc_2025-05-11_to_2025-06-26_5m.csv
  2025-05-11 00:00:00 ～ 2025-06-25 23:55:00

▶ 15m を取得中: 2025-05-01 ～ 2025-06-24
✅ 5184 行取得済: ohlcv_data/adausdc_2025-05-01_to_2025-06-24_15m.csv
  2025-05-01 00:00:00 ～ 2025-06-23 23:45:00

▶ 1h を取得中: 2025-03-24 ～ 2025-06-24
✅ 2208 行取得済: ohlcv_data/adausdc_2025-03-24_to_2025-06-24_1h.csv
  2025-03-24 00:00:00 ～ 2025-06-23 23:00:00

▶ 4h を取得中: 2024-12-21 ～ 2025-06-21
✅ 1092 行取得済: ohlcv_data/adausdc_2024-12-21_to_2025-06-21_4h.csv
  2024-12-21 00:00:00 ～ 2025-06-20 20:00:00

▶ 1d を取得中: 2024-06-17 ～ 2025-06-17
✅ 365 行取得済: ohlcv_data/adausdc_2024-06-17_to_2025-06-17_1d.csv
  2024-06-17 00:00:00 ～ 2025-06-16 00:00:00

🎉 全時間足の取得が完了しました。
